This codebase allows you to plot your polygon points to a map to see if the data points are as they should be.



**Developers : Krishna Kafle and Saral Karki**

How to use this document:

1. Use it with API's (URL)

2. Use it with local file


In [1]:
# Import the required libraries
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO
import numpy as np

import requests # For API
import os
from dotenv import load_dotenv

load_dotenv()

# geo libraries
import geopandas as gpd
from shapely import wkt
import folium
from shapely.geometry import LineString, Point, Polygon
from shapely.wkt import dumps

# visaulization library
import matplotlib.pyplot as plt


import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

#### **With API's**


Identify what the data URL'**s** are.
For surveys done through the Kobo server, the url's can be identified from :
[Kobo API's](https://kc.humanitarianresponse.info/api/v1/data)



In [2]:
# Load data URL from environment variables
data_url = os.getenv('KOBO_DATA_URL_MEXICO')

##### Your Data collection server username , password - (Kobo credentials)

In [3]:
username = os.getenv('KOBO_USERNAME_MEXICO')
password = os.getenv('KOBO_PASSWORD_MEXICO')

In [4]:
response = requests.get(data_url, auth=(username, password))

In [5]:
# Check if the request was successful
if response.status_code == 200:
    # Use BytesIO to read the response content as an Excel file
    excel_file = BytesIO(response.content)
    df = pd.read_excel(excel_file, engine='openpyxl')  # Use the openpyxl engine for .xlsx files
    print("Data loaded into DataFrame successfully.")

else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")


Data loaded into DataFrame successfully.


#### Analysis

In [6]:
df.columns

Index(['deviceid', 'start', 'end', 'Name of Enumerator', 'Region',
       'Is this plot touching any previously sampled plot?',
       'Which land use / land cover type?',
       'Specify other land use / land cover type', 'Cropping system',
       'Is weed in the field?', 'Culture type',
       'Which crop is currently on the plot?',
       'Specify other crop (monoculture)',
       'Which crops are currently on the plot?',
       'Specify other crop (policulture)',
       'Select the maize phenological stage', 'Specify other growth stage',
       'GPS point from the plot (7 meters inside the plot)',
       '_GPS point from the plot (7 meters inside the plot)_latitude',
       '_GPS point from the plot (7 meters inside the plot)_longitude',
       '_GPS point from the plot (7 meters inside the plot)_altitude',
       '_GPS point from the plot (7 meters inside the plot)_precision',
       'Record plot polygon by tapping on each corner of the plot that you see on the map',
       'shape

##### **Only subsetting the polygon geolocation and crop columns we require**

In [7]:
# Getting the required columns only

df_geo = df[['Name of Enumerator','GPS point from the plot (7 meters inside the plot)','Record plot polygon by tapping on each corner of the plot that you see on the map','Which crop is currently on the plot?']]

# Renaming the columns to adapt to the code
df_geo.columns = ['enumerators','PlotGPS','shape','crop']

In [8]:
df_geo.head()

,enumerators,PlotGPS,shape,crop
0,Adair,19.5339837 -98.8765795 2255.60009765625 13.41,19.533823169773004 -98.8766423240304 0.0 0.0;1...,Maize (hybrid)
1,Jose,17.245929 -97.5395305 1989.68310546875 4.61,NaN,Maize (hybrid)
2,Jose,17.2475303 -97.5404753 1990.0810546875 4.708,NaN,Maize (landrace) - sown manually
3,Jose,17.2483921 -97.5404571 1979.17041015625 4.835,NaN,NaN
4,Jose,17.24897 -97.5400885 1979.87158203125 3.94,NaN,Bean


In [9]:
def wkt_point(pon):
    point = "POINT ({} {})".format(pon[1], pon[0])
    return point

In [10]:
df_geo['point_geom'] = df_geo['PlotGPS'].apply(lambda x: f"POINT ({float(str(x).split()[1])} {float(str(x).split()[0])})" if pd.notna(x) and len(str(x).split()) >= 2 else np.nan)

# Fix: Only apply wkt.loads to non-null strings
df_geo['point_geometry'] = df_geo['point_geom'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
df_geo.drop('point_geom', axis=1, inplace=True) #Drop WKT column

# Create GeoDataFrame, potentially dropping null geometries if needed for plotting
gdf_point = gpd.GeoDataFrame(df_geo, geometry='point_geometry')

In [11]:
# https://python.hotexamples.com/examples/shapely.wkt/-/dumps/python-dumps-function-examples.html#0x346e25d2de439ed401c723d3f6e3e4c911cb28698b845c5bec33796af1b082ac-106,,134,
# https://github.com/Cadasta/cadasta-platform/blob/master/cadasta/xforms/utils.py

def odk_geom_to_wkt(coords):
    """Convert geometries in ODK format to WKT."""

    if coords == '':
        return ''
#     print(coords)
    if str(coords)!='nan':
        coords = coords.replace('\n', '')
        coords = coords.split(';')
        coords = [c.strip() for c in coords]
        if (coords[-1] == ''):
            coords.pop()

        if len(coords) > 1:
            # check for a geoshape taking into account
            # the bug in odk where the second coordinate in a geoshape
            # is the same as the last (first and last should be equal)
            if len(coords) > 3:
                if coords[1] == coords[-1]:  # geom is closed
                    coords.pop()
                    coords.append(coords[0])
            points = []
            for coord in coords:
                coord = coord.split(' ')
                coord = [x for x in coord if x]
                latlng = [float(coord[1]),
                          float(coord[0])]
                points.append(tuple(latlng))
            if (coords[0] != coords[-1] or len(coords) == 2):
                return dumps(LineString(points))
            else:
                return dumps(Polygon(points))
        else:
            latlng = coords[0].split(' ')
            latlng = [x for x in latlng if x]
            return dumps(Point(float(latlng[1]), float(latlng[0])))
    else:
        return np.nan

In [12]:
def wkt_loads(x):
    try:
        return wkt.loads(x)
    except Exception:
        return None

In [13]:
df_geo['geom_poly'] = df_geo['shape'].map(odk_geom_to_wkt)


In [14]:
df_geo['crop'].unique()

<StringArray>
[                      'Maize (hybrid)',
     'Maize (landrace) - sown manually',
                                    nan,
                                 'Bean',
                                'Other',
 'Maize (landrace) - sown mechanically',
                               'Forage',
                                'Wheat',
            'Maize (landrace) - Cajete',
                               'Barley',
                              'Pumpkin',
                                  'Oat']
Length: 12, dtype: str

In [15]:
df_new = df_geo
df_new['geometry'] = df_new.geom_poly.apply(wkt_loads)
df_new.drop('geom_poly', axis=1, inplace=True) #Drop WKT column
df_new = df_new.dropna(subset=['geometry'])

# Geopandas GeoDataFrame
gdf_poly = gpd.GeoDataFrame(df_new, geometry='geometry')

##### Change the map location to point to the geographic region you are on. Currently pointing to Nepal

In [16]:
# Define the location for Mexico (e.g., Mexico City)
map_location = [17.1983, -97.5745]
map_vis = folium.Map(location=map_location, zoom_start=10, tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
        attr="Google",
        name="Google Satellite")

In [17]:
# Filter out rows where point_geometry is None before plotting
gdf_point_valid = gdf_point[gdf_point.point_geometry.notnull()].reset_index(drop=True)

# Create a geometry list from the valid points only
geo_df_list = [[point.xy[1][0], point.xy[0][0]] for point in gdf_point_valid.point_geometry]

# Iterate through list and add a marker for each location
for i, coordinates in enumerate(geo_df_list):
    # assign a color marker based on crop type
    crop_type = str(gdf_point_valid.crop[i]).lower()
    if crop_type == "wheat":
        type_color = "green"
    elif crop_type == "mustard":
        type_color = "blue"
    elif crop_type == "lentil":
        type_color = "orange"
    elif crop_type == "vegetable":
        type_color = "pink"
    else:
        type_color = "purple"

    # Place the markers with the popup labels and data
    map_vis.add_child(
        folium.Marker(
            location=coordinates,
            popup="Crop: " + str(gdf_point_valid.crop[i]) + "<br>" + "Enumerator: " + str(gdf_point_valid.enumerators[i]),
            icon=folium.Icon(color=type_color)
        )
    )

In [18]:
for _, r in gdf_poly.iterrows():
    # Without simplifying the representation of each borough,
    # the map might not be displayed
    sim_geo = gpd.GeoSeries(r['geometry']).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j,
                           style_function=lambda x: {'fillColor': 'orange'})
    folium.Popup(r['enumerators']).add_to(geo_j)
    geo_j.add_to(map_vis)

In [35]:
# Display the map
map_vis

#### Download the dynamic maps locally

In [ ]:
map_vis.save("crop_map_with_polygons.html")
from google.colab import files
files.download("crop_map_with_polygons.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>